<a href="https://colab.research.google.com/github/BOMS-08/SAC-BRVM/blob/main/SAC_BRVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================================================================
# Entraînement SAC_BRVM Hebdomadaire (v3)
# 1. CVaR selon Rockafellar (pertes positives, ζ, somme des max)
# 2. ent_coef fixé à 0.01 (exploration maintenue)
# 3. Seuil d'exécution augmenté à 10 %
# 4. Fenêtre CVaR portée à 30 semaines
# ===================================================================

import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
import torch
import os
import random
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv

# ===================================================================
# 1. GRAINES ET REPRODUCTIBILITÉ
# ===================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ===================================================================
# 2. CHARGEMENT ET AGRÉGATION HEBDOMADAIRE
# ===================================================================
data_dir = '/content/drive/MyDrive/Projet_BRVM_SAC/data_processed'
model_dir = '/content/drive/MyDrive/Projet_BRVM_SAC/models'
os.makedirs(model_dir, exist_ok=True)

print("🔄 Chargement des données quotidiennes et agrégation hebdomadaire...")

# 2.1. Paramètres de normalisation quotidiens (pour filtrer colonnes)
norm_params_daily = np.load(f"{model_dir}/norm_params.npy", allow_pickle=True).item()
valid_cols = norm_params_daily['mean'].index.tolist()

# 2.2. Chargement des rendements quotidiens
train_rendements_daily = pd.read_csv(f"{data_dir}/train_rendements.csv", index_col=0, parse_dates=True)
train_rendements_daily = train_rendements_daily[valid_cols].fillna(0.0)

# 2.3. Masque quotidien
train_masque_daily = pd.read_csv(f"{data_dir}/train_masque.csv", index_col=0, parse_dates=True)
train_masque_daily = train_masque_daily[valid_cols].fillna(0.0)

# 2.4. ADTV quotidien
adtv_daily = pd.read_csv(f"{data_dir}/matrice_adtv_valeur_20d.csv", index_col=0, parse_dates=True)
adtv_daily = adtv_daily[valid_cols].fillna(1e6)

# 2.5. Régimes MS-GARCH (optionnel)
use_regimes = False   # Mettre à False pour ne pas utiliser les régimes
if use_regimes:
    regimes_daily = pd.read_csv(f"{data_dir}/regimes_msgarch.csv", index_col=0, parse_dates=True)
    regimes_daily = regimes_daily[['prob_calme', 'prob_modere', 'prob_crise']]
else:
    regimes_daily = None

# 2.6. Fonction d'agrégation hebdomadaire
def aggregate_weekly(df_returns, df_mask=None, df_adtv=None, df_regimes=None):
    """
    Agrège les données quotidiennes en hebdomadaires.
    - Rendement : produit des (1+r) - 1
    - Masque : dernier jour de la semaine
    - ADTV : moyenne hebdomadaire
    - Régimes : moyenne hebdomadaire des probabilités
    """
    df_returns = df_returns.sort_index()
    if df_mask is not None:
        df_mask = df_mask.sort_index()
    if df_adtv is not None:
        df_adtv = df_adtv.sort_index()
    if df_regimes is not None:
        df_regimes = df_regimes.sort_index()

    weekly_returns = df_returns.resample('W-FRI').apply(lambda x: (1 + x).prod() - 1)

    if df_mask is not None:
        weekly_mask = df_mask.resample('W-FRI').last()
    else:
        weekly_mask = None

    if df_adtv is not None:
        weekly_adtv = df_adtv.resample('W-FRI').mean()
    else:
        weekly_adtv = None

    if df_regimes is not None:
        weekly_regimes = df_regimes.resample('W-FRI').mean()
    else:
        weekly_regimes = None

    common_index = weekly_returns.index
    if weekly_mask is not None:
        common_index = common_index.intersection(weekly_mask.index)
    if weekly_adtv is not None:
        common_index = common_index.intersection(weekly_adtv.index)
    if weekly_regimes is not None:
        common_index = common_index.intersection(weekly_regimes.index)

    weekly_returns = weekly_returns.loc[common_index]
    if weekly_mask is not None:
        weekly_mask = weekly_mask.loc[common_index]
    if weekly_adtv is not None:
        weekly_adtv = weekly_adtv.loc[common_index]
    if weekly_regimes is not None:
        weekly_regimes = weekly_regimes.loc[common_index]

    return weekly_returns, weekly_mask, weekly_adtv, weekly_regimes

print("📊 Agrégation hebdomadaire des données d'entraînement...")
train_returns_weekly, train_mask_weekly, train_adtv_weekly, train_regimes_weekly = aggregate_weekly(
    train_rendements_daily, train_masque_daily, adtv_daily,
    regimes_daily if use_regimes else None
)

# 2.7. Normalisation hebdomadaire
mean_weekly = train_returns_weekly.mean()
std_weekly = train_returns_weekly.std().replace(0, 1e-8)
norm_params_weekly = {'mean': mean_weekly, 'std': std_weekly}
np.save(f"{model_dir}/norm_params_weekly_v3.npy", norm_params_weekly, allow_pickle=True)

train_returns_norm = (train_returns_weekly - mean_weekly) / std_weekly
train_returns_norm = train_returns_norm.fillna(0).replace([np.inf, -np.inf], 0)

print(f"✅ Données hebdomadaires prêtes : {train_returns_weekly.shape[1]} actifs, "
      f"{len(train_returns_weekly)} semaines d'entraînement.")

# ===================================================================
# 3. ENVIRONNEMENT GYMNASIUM AMÉLIORÉ (v3)
# ===================================================================
class BRVMWeeklyEnvV3(gym.Env):
    """
    Environnement hebdomadaire optimisé :
    - CVaR calculée selon Rockafellar (pertes positives)
    - Seuil d'exécution pour limiter le turnover
    - Pénalité supplémentaire sur le turnover
    - Liquidité renforcée
    - Option régimes
    """
    def __init__(self,
                 rendements_norm,
                 rendements_raw,
                 masque,
                 adtv,
                 regimes=None,
                 window_size=20,
                 transaction_cost=0.014,
                 rebalance_threshold=0.10,   # seuil relevé à 10%
                 lambda_turnover=0.001,
                 theta=0.05,
                 lambda_zou=0.05,
                 lambda_cvar=0.05,
                 alpha_cvar=0.05,
                 cvar_window=30,             # fenêtre CVaR étendue à 30 semaines
                 tpv_initial=1_000_000):
        super(BRVMWeeklyEnvV3, self).__init__()

        self.rendements_norm = rendements_norm.values
        self.rendements_raw = rendements_raw.values
        self.masque = masque.values
        self.adtv = adtv.values
        self.regimes = regimes.values if regimes is not None else None

        self.window_size = window_size
        self.c = transaction_cost
        self.rebalance_threshold = rebalance_threshold
        self.lambda_turnover = lambda_turnover
        self.theta = theta
        self.lambda_zou = lambda_zou
        self.lambda_cvar = lambda_cvar
        self.alpha_cvar = alpha_cvar
        self.cvar_window = cvar_window
        self.tpv = tpv_initial

        self.n_steps, self.n_assets = self.rendements_norm.shape
        self.current_step = self.window_size

        self.action_space = spaces.Box(low=-1, high=1, shape=(self.n_assets,), dtype=np.float32)

        obs_dim = (self.window_size * self.n_assets) + self.n_assets
        if self.regimes is not None:
            obs_dim += self.regimes.shape[1]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self.weights = np.ones(self.n_assets, dtype=np.float32) / self.n_assets
        self.historique_rendements = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.weights = np.ones(self.n_assets, dtype=np.float32) / self.n_assets
        self.historique_rendements = []
        return self._get_observation(), {}

    def _get_observation(self):
        start = self.current_step - self.window_size
        end = self.current_step
        past_norm = self.rendements_norm[start:end].flatten()
        past_norm = np.nan_to_num(past_norm, nan=0.0)
        obs = np.concatenate([past_norm, self.weights])
        if self.regimes is not None:
            prob_t = self.regimes[self.current_step]
            obs = np.concatenate([obs, prob_t])
        obs = obs.astype(np.float32)
        return np.nan_to_num(obs, nan=0.0)

    def step(self, action):
        # 1. Softmax masqué
        action = np.clip(action, -5, 5)
        mask_t = self.masque[self.current_step]
        exp_a = np.exp(action - np.max(action)) * mask_t
        if exp_a.sum() > 1e-6:
            target_weights = exp_a / exp_a.sum()
        else:
            target_weights = self.weights.copy()
        target_weights = np.nan_to_num(target_weights, nan=0.0)
        if np.sum(target_weights) == 0:
            target_weights = np.ones(self.n_assets) / self.n_assets

        # 2. Seuil d'exécution (10% désormais)
        potential_turnover = np.sum(np.abs(target_weights - self.weights))
        if potential_turnover >= self.rebalance_threshold:
            new_weights = target_weights.copy()
            turnover_reel = potential_turnover
            transaction_fee = self.c * turnover_reel
        else:
            new_weights = self.weights.copy()
            turnover_reel = 0.0
            transaction_fee = 0.0

        # 3. Rendement net
        r_raw = self.rendements_raw[self.current_step] * mask_t
        portfolio_return = np.sum(new_weights * r_raw)
        net_return = portfolio_return - transaction_fee

        # 4. Pénalité liquidité (Zou)
        adtv_t = self.adtv[self.current_step] + 1e-8
        position_valeur = new_weights * self.tpv
        seuil = self.theta * adtv_t
        depassement = np.maximum(0, (position_valeur - seuil) / self.tpv)
        zou_penalty = self.lambda_zou * np.sum(depassement ** 2)

        # 5. Pénalité CVaR selon Rockafellar (corrigée et étendue)
        self.historique_rendements.append(net_return)
        if len(self.historique_rendements) > self.cvar_window:
            self.historique_rendements.pop(0)

        if len(self.historique_rendements) >= max(10, self.cvar_window // 2):
            # Convertir les rendements en pertes positives
            pertes = [-r for r in self.historique_rendements]
            # ζ = VaR historique au niveau 1-alpha
            zeta = np.percentile(pertes, 100 * (1 - self.alpha_cvar))
            # Somme des excès au-delà de ζ
            S = len(pertes)
            somme_pertes_extremes = sum(max(0.0, p - zeta) for p in pertes)
            # CVaR de Rockafellar
            cvar = zeta + (1 / (1 - self.alpha_cvar)) * (1 / S) * somme_pertes_extremes
        else:
            cvar = 0.0

        cvar_penalty = self.lambda_cvar * max(0.0, cvar)

        # 6. Récompense finale
        reward = net_return - zou_penalty - cvar_penalty - self.lambda_turnover * turnover_reel

        # 7. Mise à jour
        self.weights = new_weights
        self.current_step += 1
        done = self.current_step >= self.n_steps - 1

        info = {
            'portfolio_return': net_return,
            'turnover': turnover_reel,
            'zou_penalty': zou_penalty,
            'cvar_penalty': cvar_penalty
        }

        return self._get_observation(), float(reward), done, False, info

# ===================================================================
# 4. INSTANCIATION DE L'ENVIRONNEMENT
# ===================================================================
print("\n🤖 Création de l'environnement SAC_BRVM hebdomadaire v3...")

env_kwargs = dict(
    rendements_norm=train_returns_norm,
    rendements_raw=train_returns_weekly,
    masque=train_mask_weekly,
    adtv=train_adtv_weekly,
    regimes=train_regimes_weekly if use_regimes else None,
    window_size=20,
    transaction_cost=0.014,
    rebalance_threshold=0.10,   # seuil augmenté à 10%
    lambda_turnover=0.001,
    theta=0.05,
    lambda_zou=0.05,
    lambda_cvar=0.05,
    alpha_cvar=0.05,
    cvar_window=30,             # fenêtre CVaR élargie
    tpv_initial=1_000_000
)

env = BRVMWeeklyEnvV3(**env_kwargs)

# Test de l'environnement
obs, _ = env.reset()
assert not np.any(np.isnan(obs)), "❌ Observation contient des NaN"
action = env.action_space.sample()
obs, reward, done, _, info = env.step(action)
assert not np.any(np.isnan(obs)), "❌ Observation après step contient des NaN"
print("✅ Environnement validé avec succès.")

vec_env = DummyVecEnv([lambda: env])

# ===================================================================
# 5. ENTRAÎNEMENT
# ===================================================================
print("\n🚀 Démarrage de l'entraînement SAC_BRVM hebdo v3 (ent_coef=0.01)...")

model = SAC(
    policy="MlpPolicy",
    env=vec_env,
    learning_rate=3e-4,
    buffer_size=100000,
    batch_size=256,
    ent_coef=0.01,              # exploration fixe
    gamma=0.99,
    verbose=1,
    tensorboard_log="./sac_brvm_weekly_v3_tensorboard/",
    seed=SEED,
    policy_kwargs=dict(net_arch=[256, 256])
)

model.learn(total_timesteps=200000, log_interval=10)

# Sauvegarde
save_path = f"{model_dir}/sac_brvm_weekly_v3"
model.save(save_path)

print(f"\n✅ ENTRAÎNEMENT TERMINÉ. MODÈLE SAUVEGARDÉ DANS : {save_path}.zip")
print("📊 TensorBoard logs disponibles dans ./sac_brvm_weekly_v3_tensorboard/")